In [1]:
import pandas as pd

# 1. Chargement du fichier (attention au séparateur, souvent ';' pour l'INSEE)
df_filosofi = pd.read_csv('DS_FILOSOFI_CC_2023_data.csv', sep=';', low_memory=False)

# 2. On affiche les premières lignes et les noms des colonnes
print("--- Aperçu des données Filosofi ---")
display(df_filosofi.head())

print("\n--- Liste des colonnes disponibles ---")
print(df_filosofi.columns.tolist())

# 3. Vérification rapide des types
print("\n--- Types des colonnes clés ---")
# On cherche CODGEO (le code commune) et MED21 ou MED23
cols_interet = [col for col in df_filosofi.columns if 'CODGEO' in col or 'MED' in col]
print(df_filosofi[cols_interet].dtypes)

--- Aperçu des données Filosofi ---


,FILOSOFI_MEASURE,GEO,GEO_OBJECT,UNIT_MEASURE,CONF_STATUS,OBS_STATUS,UNIT_MULT,TIME_PERIOD,OBS_VALUE
0,S_EI_DI_UNE,EPT_200057966,OTHER,PT,F,O,0,2023,NaN
1,D3_SL,EPT_200057941,OTHER,EUR_YR,F,O,0,2023,NaN
2,D2_SL,EPT_200057875,OTHER,EUR_YR,F,O,0,2023,NaN
3,IQR_SL,EPT_200058790,OTHER,EUR_YR,F,O,0,2023,NaN
4,S_RET_PEN_DI,EPT_200057875,OTHER,PT,F,O,0,2023,NaN



--- Liste des colonnes disponibles ---
['FILOSOFI_MEASURE', 'GEO', 'GEO_OBJECT', 'UNIT_MEASURE', 'CONF_STATUS', 'OBS_STATUS', 'UNIT_MULT', 'TIME_PERIOD', 'OBS_VALUE']

--- Types des colonnes clés ---
Series([], dtype: object)


## Fichier initial poru récup code commune

In [6]:
import pandas as pd

# 1. Chargement complet (ou partiel si le fichier est énorme)
# On force déjà le type str sur les colonnes identifiées comme codes géographiques
df_dvf = pd.read_csv('dvf_final_2020_2025.csv', dtype={'code_commune': str, 'code_departement': str}, low_memory=False)

# 2. Correction immédiate du code commune (le fameux zfill pour passer de "1130" à "01130")
if 'code_commune' in df_dvf.columns:
    df_dvf['code_commune'] = df_dvf['code_commune'].str.zfill(5)
    print("✓ Colonne 'code_commune' formatée sur 5 caractères.")
else:
    print("⚠️ Attention : La colonne 'code_commune' n'a pas été trouvée sous ce nom exact.")

# 3. Vérification des données "plus loin" dans le fichier
print("\n--- Aperçu des lignes 10 000 à 10 005 ---")
display(df_dvf[['code_commune', 'valeur_fonciere', 'type_local']].iloc[10000:10005])

# 4. Check rapide sur les départements présents
print("\nTop 5 des départements avec le plus de transactions :")
print(df_dvf['code_departement'].value_counts().head(5))

✓ Colonne 'code_commune' formatée sur 5 caractères.

--- Aperçu des lignes 10 000 à 10 005 ---


,code_commune,valeur_fonciere,type_local
10000,02734,210000.0,Maison
10001,02439,250000.0,Maison
10002,02810,43000.0,Appartement
10003,02551,130000.0,Maison
10004,02148,145000.0,Maison



Top 5 des départements avec le plus de transactions :
code_departement
59    206250
13    170509
83    161457
75    157355
06    152779
Name: count, dtype: int64


In [15]:
# 1. Calcul des manquants sur le fichier source
missing_dvf = df_dvf.isnull().sum()
pct_dvf = (df_dvf.isnull().sum() / len(df_dvf)) * 100

# 2. Focus sur les colonnes clés pour la fusion
cols_cles = ['code_commune', 'latitude', 'longitude', 'valeur_fonciere', 'surface_reelle_bati']
dvf_analysis = pd.DataFrame({
    'Manquants': missing_dvf[cols_cles],
    'Pourcentage (%)': pct_dvf[cols_cles]
})

print("--- Diagnostic du fichier SOURCE (df_dvf) ---")
display(dvf_analysis)

# 3. Vérification du format du code_commune
if 'code_commune' in df_dvf.columns:
    print(f"\nType de la colonne code_commune : {df_dvf['code_commune'].dtype}")
    print(f"Exemple de valeurs : {df_dvf['code_commune'].dropna().unique()[:5]}")

--- Diagnostic du fichier SOURCE (df_dvf) ---


,Manquants,Pourcentage (%)
code_commune,0,0.000000
latitude,104520,1.793362
longitude,104520,1.793362
valeur_fonciere,0,0.000000
surface_reelle_bati,0,0.000000



Type de la colonne code_commune : str
Exemple de valeurs : <StringArray>
['01130', '01451', '01364', '01053', '01177']
Length: 5, dtype: str


In [7]:
# On s'assure que tout est en texte et sur 5 caractères
df_dvf['code_commune'] = df_dvf['code_commune'].astype(str).str.zfill(5)

# Vérification sur les départements 1 à 9
print("Vérification des codes pour l'Ain (01) ou l'Aisne (02) :")
print(df_dvf[df_dvf['code_commune'].str.startswith('0')]['code_commune'].unique()[:5])

Vérification des codes pour l'Ain (01) ou l'Aisne (02) :
<StringArray>
['01130', '01451', '01364', '01053', '01177']
Length: 5, dtype: str


## Integration des codes comunes dans le fichier ML + ecole  + commerce

In [8]:
# Chargement du fichier BPE
df_bpe = pd.read_csv('dvf_bpe_ecole_commerce_sante.csv', low_memory=False)

# Vérification de sécurité avant fusion
if len(df_bpe) == len(df_dvf):
    print("✅ Alignement parfait. Intégration du code_commune via l'index.")
    df_bpe['code_commune'] = df_dvf['code_commune'].values
else:
    print(f"⚠️ Décalage détecté : DVF ({len(df_dvf)} lignes) vs BPE ({len(df_bpe)} lignes).")
    print("Nous devrons utiliser une colonne commune (ID) pour sécuriser la donnée.")

⚠️ Décalage détecté : DVF (5828160 lignes) vs BPE (4640555 lignes).
Nous devrons utiliser une colonne commune (ID) pour sécuriser la donnée.


In [9]:
# Lister les colonnes de chaque fichier
cols_dvf = set(df_dvf.columns)
cols_bpe = set(pd.read_csv('dvf_bpe_ecole_commerce_sante.csv', nrows=1).columns)

# Voir s'il y a des colonnes identiques
communes = cols_dvf.intersection(cols_bpe)
print(f"Colonnes communes trouvées : {communes}")

Colonnes communes trouvées : {'surface_reelle_bati', 'latitude', 'annee', 'nombre_pieces_principales', 'mois', 'code_departement', 'surface_terrain', 'longitude', 'valeur_fonciere'}


In [16]:
# 1. Préparation des clés (on arrondit pour être sûr)
df_dvf['latitude_round'] = df_dvf['latitude'].round(4)
df_dvf['longitude_round'] = df_dvf['longitude'].round(4)
df_bpe['latitude_round'] = df_bpe['latitude'].round(4)
df_bpe['longitude_round'] = df_bpe['longitude'].round(4)

# 2. On crée un référentiel plus simple
# On utilise Latitude, Longitude et Valeur Foncière (c'est déjà une signature très forte)
cles_fusion_simple = ['latitude_round', 'longitude_round', 'valeur_fonciere']

df_ref_geo_simple = df_dvf[cles_fusion_simple + ['code_commune']].drop_duplicates(subset=cles_fusion_simple)

# 3. On nettoie df_bpe de l'ancienne colonne ratée
if 'code_commune' in df_bpe.columns:
    df_bpe = df_bpe.drop(columns=['code_commune'])

# 4. Nouvelle fusion
df_bpe = pd.merge(
    df_bpe, 
    df_ref_geo_simple, 
    on=cles_fusion_simple, 
    how='left'
)

# 5. Diagnostic final
nouveaux_manquants = df_bpe['code_commune'].isna().sum()
print(f"Nombre de manquants après fusion simplifiée : {nouveaux_manquants}")
print(f"Soit {(nouveaux_manquants/len(df_bpe))*100:.2f}% d'échec.")


Nombre de manquants après fusion simplifiée : 0
Soit 0.00% d'échec.


In [17]:
# 1. Calcul du nombre et du pourcentage de manquants par colonne
missing_count = df_bpe.isnull().sum()
missing_percentage = (df_bpe.isnull().sum() / len(df_bpe)) * 100

# 2. Création d'un DataFrame de synthèse pour une lecture facile
missing_df = pd.DataFrame({
    'Valeurs Manquantes': missing_count,
    'Pourcentage (%)': missing_percentage
})

# 3. On trie pour voir les colonnes les plus problématiques en haut
missing_df = missing_df.sort_values(by='Pourcentage (%)', ascending=False)

print("--- Analyse des données manquantes dans df_bpe ---")
display(missing_df)

# 4. Optionnel : Visualisation rapide si tu as beaucoup de colonnes
if missing_percentage.max() > 0:
    print("\nVisualisation des colonnes ayant des manquants :")
    missing_df[missing_df['Valeurs Manquantes'] > 0]['Pourcentage (%)'].plot(kind='barh', figsize=(10, 6))

--- Analyse des données manquantes dans df_bpe ---


,Valeurs Manquantes,Pourcentage (%)
surface_reelle_bati,0,0.0
nombre_pieces_principales,0,0.0
surface_terrain,0,0.0
valeur_fonciere,0,0.0
annee,0,0.0
mois,0,0.0
code_departement,0,0.0
longitude,0,0.0
latitude,0,0.0
is_maison,0,0.0


In [18]:
# Sauvegarde du fichier consolidé
df_bpe.to_csv('dvf_bpe_ecole_commerce_sante_codecommune.csv', index=False)

print("✅ Fichier sauvegardé avec succès sous le nom : dvf_bpe_ecole_commerce_sante_codecommune.csv")

✅ Fichier sauvegardé avec succès sous le nom : dvf_bpe_ecole_commerce_sante_codecommune.csv


## Recup des 2 fichiers pour fusion avec fichier Filosophi


In [2]:
import pandas as pd

# 1. Recharger ton fichier consolidé
df_bpe = pd.read_csv('dvf_bpe_ecole_commerce_sante_codecommune.csv', dtype={'code_commune': str})

# 2. Charger un aperçu de FILOSOFI pour l'inspecter
# Note : Les fichiers INSEE utilisent souvent le point-virgule ';' comme séparateur
df_filo_preview = pd.read_csv('FILO2021_DISP_COM.csv', sep=';', nrows=5)

print("--- Colonnes disponibles dans FILOSOFI ---")
print(df_filo_preview.columns.tolist())

# 3. Afficher les premières lignes pour vérifier le code commune et les revenus
display(df_filo_preview.head())

--- Colonnes disponibles dans FILOSOFI ---
['CODGEO', 'NBMEN21', 'NBPERS21', 'NBUC21', 'Q121', 'Q221', 'Q321', 'Q3_Q1', 'D121', 'D221', 'D321', 'D421', 'D621', 'D721', 'D821', 'D921', 'RD', 'S80S2021', 'GI21', 'PACT21', 'PTSA21', 'PCHO21', 'PBEN21', 'PPEN21', 'PPAT21', 'PPSOC21', 'PPFAM21', 'PPMINI21', 'PPLOGT21', 'PIMPOT21', 'AGE1Q121', 'AGE1Q221', 'AGE1Q321', 'AGE1Q3_Q1', 'AGE1D121', 'AGE1D221', 'AGE1D321', 'AGE1D421', 'AGE1D621', 'AGE1D721', 'AGE1D821', 'AGE1D921', 'AGE1RD', 'AGE1S80S2021', 'AGE1GI21', 'AGE1PACT21', 'AGE1PTSA21', 'AGE1PCHO21', 'AGE1PBEN21', 'AGE1PPEN21', 'AGE1PPAT21', 'AGE1PPSOC21', 'AGE1PPFAM21', 'AGE1PPMINI21', 'AGE1PPLOGT21', 'AGE1PIMPOT21', 'AGE2Q121', 'AGE2Q221', 'AGE2Q321', 'AGE2Q3_Q1', 'AGE2D121', 'AGE2D221', 'AGE2D321', 'AGE2D421', 'AGE2D621', 'AGE2D721', 'AGE2D821', 'AGE2D921', 'AGE2RD', 'AGE2S80S2021', 'AGE2GI21', 'AGE2PACT21', 'AGE2PTSA21', 'AGE2PCHO21', 'AGE2PBEN21', 'AGE2PPEN21', 'AGE2PPAT21', 'AGE2PPSOC21', 'AGE2PPFAM21', 'AGE2PPMINI21', 'AGE2PPLOGT21'

,CODGEO,NBMEN21,NBPERS21,NBUC21,Q121,Q221,Q321,Q3_Q1,D121,D221,...,OPR6PTSA21,OPR6PCHO21,OPR6PBEN21,OPR6PPEN21,OPR6PPAT21,OPR6PPSOC21,OPR6PPFAM21,OPR6PPMINI21,OPR6PPLOGT21,OPR6PIMPOT21
0,1001,346,895,"590,8",s,25820,s,s,s,s,...,s,s,s,s,s,s,s,s,s,s
1,1002,115,266,"181,0",s,24480,s,s,s,s,...,s,s,s,s,s,s,s,s,s,s
2,1004,6855,15092,"10398,2",15800,21660,28430,12630,11890,14640,...,s,s,s,s,s,s,s,s,s,s
3,1005,800,2028,"1329,7",20010,24610,31180,11170,15560,18980,...,s,s,s,s,s,s,s,s,s,s
4,1006,51,107,"76,6",s,24210,s,s,s,s,...,s,s,s,s,s,s,s,s,s,s


In [4]:
import pandas as pd
import numpy as np

# 1. Chargement du fichier complet (assure-toi que le nom du fichier est exact)
# On utilise df_filo_preview comme tu l'as suggéré
df_filo_preview = pd.read_csv('FILO2021_DISP_COM.csv', sep=';', dtype={'CODGEO': str})

# 2. Sélection des colonnes utiles
cols_utiles = ['CODGEO', 'Q221', 'NBPERS21', 'PIMPOT21', 'PPSOC21']
df_filo_clean = df_filo_preview[cols_utiles].copy()

# 3. Nettoyage des "s" (Secret Statistique)
# On convertit en nombres : les chiffres restent, les "s" deviennent NaN
for col in ['Q221', 'NBPERS21', 'PIMPOT21', 'PPSOC21']:
    df_filo_clean[col] = pd.to_numeric(df_filo_clean[col], errors='coerce')

# 4. Renommage pour plus de clarté
df_filo_clean = df_filo_clean.rename(columns={
    'Q221': 'revenu_median',
    'PIMPOT21': 'pct_menages_imposes',
    'PPSOC21': 'pct_aides_sociales',
    'NBPERS21': 'population_commune'
})

print("Nettoyage réussi !")
display(df_filo_clean.head())

Nettoyage réussi !


,CODGEO,revenu_median,population_commune,pct_menages_imposes,pct_aides_sociales
0,01001,25820.0,895.0,NaN,NaN
1,01002,24480.0,266.0,NaN,NaN
2,01004,21660.0,15092.0,NaN,NaN
3,01005,24610.0,2028.0,NaN,NaN
4,01006,24210.0,107.0,NaN,NaN


In [5]:
# On regarde le nombre de valeurs non vides pour chaque colonne
print("--- Analyse de la complétude des données ---")
info_colonnes = df_filo_clean[['revenu_median', 'pct_menages_imposes', 'pct_aides_sociales']].notna().sum()
pourcentage_remplissage = (info_colonnes / len(df_filo_clean)) * 100

summary = pd.DataFrame({
    'Valeurs présentes': info_colonnes,
    'Taux de remplissage (%)': pourcentage_remplissage
})

display(summary)

# On regarde aussi si on a des valeurs différentes de 0
print("\n--- Statistiques descriptives (pour vérifier si c'est utile) ---")
display(df_filo_clean[['pct_menages_imposes', 'pct_aides_sociales']].describe())

--- Analyse de la complétude des données ---


,Valeurs présentes,Taux de remplissage (%)
revenu_median,31325,89.681926
pct_menages_imposes,0,0.000000
pct_aides_sociales,0,0.000000



--- Statistiques descriptives (pour vérifier si c'est utile) ---


,pct_menages_imposes,pct_aides_sociales
count,0.0,0.0
mean,NaN,NaN
std,NaN,NaN
min,NaN,NaN
25%,NaN,NaN
50%,NaN,NaN
75%,NaN,NaN
max,NaN,NaN


In [6]:
# 1. On ne garde que ce qui fonctionne : Code Géo et Revenu Médian
df_filo_final = df_filo_clean[['CODGEO', 'revenu_median', 'population_commune']].copy()

# 2. On supprime les lignes totalement vides de revenus si nécessaire (optionnel)
# Mais on va plutôt fusionner d'abord
df_immobilier = pd.read_csv('dvf_bpe_ecole_commerce_sante_codecommune.csv', dtype={'code_commune': str})

# 3. Fusion finale
df_final_ml = pd.merge(
    df_immobilier, 
    df_filo_final, 
    left_on='code_commune', 
    right_on='CODGEO', 
    how='left'
)

# 4. Nettoyage de la colonne en double
df_final_ml.drop(columns=['CODGEO'], inplace=True)

# 5. On remplit les 10% de revenus manquants par la médiane globale (pour ne pas avoir de trous)
valeur_remplacement = df_final_ml['revenu_median'].median()
df_final_ml['revenu_median'] = df_final_ml['revenu_median'].fillna(valeur_remplacement)

print(f"✅ Dataset prêt ! Nombre de lignes : {len(df_final_ml)}")
print(f"Vérification des manquants sur le revenu : {df_final_ml['revenu_median'].isna().sum()}")

✅ Dataset prêt ! Nombre de lignes : 4640555
Vérification des manquants sur le revenu : 0


In [7]:
df_final_ml.to_csv('DATASET_PRET_POUR_MACHINE_LEARNING.csv', index=False)
print("💾 Fichier sauvegardé. Prêt pour l'entraînement !")

💾 Fichier sauvegardé. Prêt pour l'entraînement !
